# Ensemble Classification Pipeline Example

This notebook demonstrates how to use the trained domain classification pipeline to make predictions.  
It supports both label prediction and probability estimation, with optional SHAP explanations.  
The first section shows how to classify a small batch of domains interactively;  
the second one computes performance metrics across the entire test dataset.


In [ ]:
# Import necessary modules
import sys

from core.validator import load_saved_split, load_train_split, load_random_sample
from pipeline import DomainClassifier


### Set label and dataset

In [ ]:
MALICIOUS_LABEL = "phishing"  # phishing / malware               # 1 / 2 / 3
VERIFICATION = True           # True / False, use verification dataset of validation dataset

### Forced sequential cascade across aligned stage splits

This section evaluates the same test samples through Stage 1, then forwards only malicious false negatives to Stage 2, and finally to Stage 3. It assumes the saved stage splits are row-aligned, which should hold if all stage subsets were created from the same original dataframe and only feature columns were dropped before calling `train_test_split(...)`.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, classification_report, f1_score

from core.validator import load_saved_split
from models.model_wrapper import ModelWrapper
from core.utils import safe_predict

# ── Configuration ─────────────────────────────────────────
ARCHITECTURES   = ["cnn", "XgBoost", "Lgbm", "feedforward", "svm"]
VERSION         = "v1.1"
MALICIOUS_LABEL = "phishing"
VERIFICATION    = False

STAGE_THRESHOLDS = {
    1: 1,   # >=2/5 votes → flag as malicious at stage 1
    2: 1,   # >=2/5 votes → flag as malicious at stage 2
    3: 1,   # >=3/5 votes → flag as malicious at stage 3 (strict majority)
}

STAGE_COLS = {1: 62, 2: 128, 3: 176}

# ── Load each stage's own dataset ─────────────────────────
# Each split was saved with the features that stage was trained on.
# Labels must be aligned (same samples, same order) across all three.
x1_test, y1_test = load_saved_split(1, MALICIOUS_LABEL, folder="./data/", verification=VERIFICATION)
x2_test, y2_test = load_saved_split(2, MALICIOUS_LABEL, folder="./data/", verification=VERIFICATION)
x3_test, y3_test = load_saved_split(3, MALICIOUS_LABEL, folder="./data/", verification=VERIFICATION)

print(f"Stage 1: {x1_test.shape}  labels: {y1_test.shape}")
print(f"Stage 2: {x2_test.shape}  labels: {y2_test.shape}")
print(f"Stage 3: {x3_test.shape}  labels: {y3_test.shape}")

assert len(y1_test) == len(y2_test) == len(y3_test), \
    "Splits have different sample counts — cannot run aligned cascade."
assert np.array_equal(y1_test, y2_test) and np.array_equal(y2_test, y3_test), \
    "Labels are not aligned across stage splits."

print("Labels aligned across all stages.\n")

def to_binary_vector(y):
    y = np.asarray(y)
    if y.dtype.kind in {"i", "u", "b"}:
        return y.astype(int)
    if y.dtype.kind == "f":
        return np.rint(y).astype(int)
    normalized = np.array([str(v).strip().lower() for v in y], dtype=object)
    positive = {"1", "true", "malicious", "phishing", "malware"}
    return np.array([1 if v in positive else 0 for v in normalized], dtype=int)


def run_base_models(stage, X):
    """Run all base models for a given stage, return vote_sum and per-arch preds."""
    model_wrapper = ModelWrapper(model_dir="./models")
    all_preds = {}
    for arch in ARCHITECTURES:
        model = model_wrapper.load(
            arch_name=arch,
            label=MALICIOUS_LABEL,
            prefix=f"stage_{stage}",
            version=VERSION,
        )
        preds = safe_predict(model, X, arch, MALICIOUS_LABEL, stage)
        all_preds[arch] = np.asarray(preds).flatten().astype(int)

    votes    = np.vstack(list(all_preds.values()))  # (n_models, n_samples)
    vote_sum = votes.sum(axis=0)                    # (n_samples,)
    return vote_sum, all_preds


# ── Run all base models up front on full datasets ──────────
# We run on the full dataset per stage so we can do threshold analysis.
# The cascade then indexes into these cached results.
y = to_binary_vector(y1_test)
n = len(y)

print("Running base models for all 3 stages...")

print(f"  Stage 1 ({x1_test.shape[1]} features, {n} samples)...")
s1_votes_all, s1_per_model = run_base_models(1, x1_test)

print(f"  Stage 2 ({x2_test.shape[1]} features, {n} samples)...")
s2_votes_all, s2_per_model = run_base_models(2, x2_test)

print(f"  Stage 3 ({x3_test.shape[1]} features, {n} samples)...")
s3_votes_all, s3_per_model = run_base_models(3, x3_test)

print("Done. All base model predictions cached.\n")


# ── Vote distribution analysis ─────────────────────────────
def print_vote_distribution(stage, vote_sum, y_true):
    print(f"\n  Stage {stage} vote distribution:")
    print(f"  {'Votes':>6}  {'Total':>7}  {'TrueMal':>8}  {'TrueBen':>8}  {'MalRate':>8}  {'CumRecall':>10}")
    total_mal = int((y_true == 1).sum())
    cum_tp = 0
    for v in range(6):
        mask     = vote_sum == v
        total    = int(mask.sum())
        true_mal = int((y_true[mask] == 1).sum())
        true_ben = int((y_true[mask] == 0).sum())
        mal_rate = true_mal / total if total > 0 else 0.0
        cum_tp  += true_mal
        cum_rec  = cum_tp / total_mal if total_mal > 0 else 0.0
        marker   = "  <-- threshold" if v == STAGE_THRESHOLDS[stage] - 1 else ""
        print(f"  {v:>6}  {total:>7}  {true_mal:>8}  {true_ben:>8}  {mal_rate:>7.1%}  {cum_rec:>9.1%}{marker}")

print("=" * 65)
print("VOTE DISTRIBUTION ANALYSIS")
print("(CumRecall = % of all malicious caught at <=this vote level)")
print("=" * 65)
print_vote_distribution(1, s1_votes_all, y)
print_vote_distribution(2, s2_votes_all, y)
print_vote_distribution(3, s3_votes_all, y)


# ── F1 vs threshold sweep ──────────────────────────────────
print("\n" + "=" * 65)
print("F1 vs VOTE THRESHOLD (standalone per stage, full dataset)")
print("=" * 65)

for stage, votes_all in [(1, s1_votes_all), (2, s2_votes_all), (3, s3_votes_all)]:
    print(f"\n  Stage {stage}:")
    print(f"  {'Threshold':>10}  {'F1-macro':>10}  {'F1-mal':>8}  {'F1-ben':>8}  {'Recall-mal':>11}  {'Prec-mal':>10}")
    for t in range(1, 6):
        pred  = (votes_all >= t).astype(int)
        f1m   = f1_score(y, pred, average="macro")
        f1mal = f1_score(y, pred, average="binary", pos_label=1)
        f1ben = f1_score(y, pred, average="binary", pos_label=0)
        tn_, fp_, fn_, tp_ = confusion_matrix(y, pred).ravel()
        rec   = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0.0
        pre   = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else 0.0
        marker = "  <-- current" if t == STAGE_THRESHOLDS[stage] else ""
        print(f"  {t:>10}  {f1m:>10.4f}  {f1mal:>8.4f}  {f1ben:>8.4f}  {rec:>11.4f}  {pre:>10.4f}{marker}")


# ── Cascade with configured thresholds ────────────────────
print("\n" + "=" * 65)
print(f"CASCADE  |  thresholds: S1>={STAGE_THRESHOLDS[1]}  S2>={STAGE_THRESHOLDS[2]}  S3>={STAGE_THRESHOLDS[3]}")
print("=" * 65)

final_pred   = np.full(n, -1, dtype=int)
stage1_votes = s1_votes_all.astype(float)
stage2_votes = np.full(n, np.nan)
stage3_votes = np.full(n, np.nan)

# Stage 1 — classify all samples using stage 1 dataset
s1_pred          = (s1_votes_all >= STAGE_THRESHOLDS[1]).astype(int)
malicious_s1_idx = np.where(s1_pred == 1)[0]
pass_to_s2_idx   = np.where(s1_pred == 0)[0]
final_pred[malicious_s1_idx] = 1
print(f"\nStage 1: {n} in → {len(malicious_s1_idx)} flagged, {len(pass_to_s2_idx)} forwarded")

# Stage 2 — classify S1 survivors using stage 2 dataset rows
malicious_s2_idx = np.array([], dtype=int)
pass_to_s3_idx   = np.array([], dtype=int)

if len(pass_to_s2_idx) > 0:
    s2_votes_sub = s2_votes_all[pass_to_s2_idx]
    stage2_votes[pass_to_s2_idx] = s2_votes_sub.astype(float)
    s2_pred          = (s2_votes_sub >= STAGE_THRESHOLDS[2]).astype(int)
    malicious_s2_idx = pass_to_s2_idx[s2_pred == 1]
    pass_to_s3_idx   = pass_to_s2_idx[s2_pred == 0]
    final_pred[malicious_s2_idx] = 1
    print(f"Stage 2: {len(pass_to_s2_idx)} in → {len(malicious_s2_idx)} flagged, {len(pass_to_s3_idx)} forwarded")

# Stage 3 — classify S2 survivors using stage 3 dataset rows
if len(pass_to_s3_idx) > 0:
    s3_votes_sub = s3_votes_all[pass_to_s3_idx]
    stage3_votes[pass_to_s3_idx] = s3_votes_sub.astype(float)
    s3_pred = (s3_votes_sub >= STAGE_THRESHOLDS[3]).astype(int)
    final_pred[pass_to_s3_idx] = s3_pred
    print(f"Stage 3: {len(pass_to_s3_idx)} in → {int(s3_pred.sum())} flagged, {int((s3_pred == 0).sum())} benign")

assert np.all(final_pred >= 0), "Some samples have no final prediction!"

# ── End-to-end metrics ─────────────────────────────────────
tn, fp, fn, tp = confusion_matrix(y, final_pred).ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = (tp + tn) / n

print(f"""
--- Routing summary ---
  Stopped  at Stage 1 : {len(malicious_s1_idx):>6}
  Stopped  at Stage 2 : {len(malicious_s2_idx):>6}
  Decided  at Stage 3 : {len(pass_to_s3_idx):>6}

--- Confusion matrix ---
  TP : {tp}   FN : {fn}
  FP : {fp}   TN : {tn}

Accuracy  : {accuracy:.4f}
Precision : {precision:.4f}
Recall    : {recall:.4f}
F1 score  : {f1:.6f}
""")
print(classification_report(y, final_pred, target_names=["benign", MALICIOUS_LABEL]))

# ── Per-stage standalone F1 (on the subset that reached each stage) ──
print("--- Per-stage standalone F1 (samples that reached that stage) ---")

print(f"\nStage 1 ({n} samples):")
print(classification_report(y, s1_pred, target_names=["benign", MALICIOUS_LABEL], digits=4))

if len(pass_to_s2_idx) > 0:
    print(f"Stage 2 ({len(pass_to_s2_idx)} samples that S1 passed as benign):")
    print(classification_report(
        y[pass_to_s2_idx],
        (s2_votes_all[pass_to_s2_idx] >= STAGE_THRESHOLDS[2]).astype(int),
        target_names=["benign", MALICIOUS_LABEL], digits=4
    ))

if len(pass_to_s3_idx) > 0:
    print(f"Stage 3 ({len(pass_to_s3_idx)} samples that S2 passed as benign):")
    print(classification_report(
        y[pass_to_s3_idx],
        (s3_votes_all[pass_to_s3_idx] >= STAGE_THRESHOLDS[3]).astype(int),
        target_names=["benign", MALICIOUS_LABEL], digits=4
    ))

# ── Results dataframe ──────────────────────────────────────
decided_at = np.zeros(n, dtype=int)
decided_at[malicious_s1_idx] = 1
decided_at[malicious_s2_idx] = 2
decided_at[pass_to_s3_idx]   = 3

results_df = pd.DataFrame({
    "sample_idx"      : np.arange(n),
    "expected"        : y,
    "final_pred"      : final_pred,
    "decided_at_stage": decided_at,
    "stage1_votes"    : stage1_votes,
    "stage2_votes"    : stage2_votes,
    "stage3_votes"    : stage3_votes,
})
results_df["correct"] = (results_df["expected"] == results_df["final_pred"]).astype(int)
display(results_df.head(20))

Stage 1: (98075, 62)  labels: (98075,)
Stage 2: (98075, 128)  labels: (98075,)
Stage 3: (98075, 176)  labels: (98075,)
Labels aligned across all stages.

Running base models for all 3 stages...
  Stage 1 (62 features, 98075 samples)...
📦 Loading model from ./models/cnn_stage_1_phishing_v1.1.keras
3065/3065 [==============================] - 3s 834us/step
📦 Loading model from ./models/XgBoost_stage_1_phishing_v1.1.xgb
📦 Loading model from ./models/Lgbm_stage_1_phishing_v1.1.pkl
📦 Loading model from ./models/feedforward_stage_1_phishing_v1.1.keras
3065/3065 [==============================] - 3s 964us/step
📦 Loading model from ./models/svm_stage_1_phishing_v1.1.pkl
  Stage 2 (128 features, 98075 samples)...
📦 Loading model from ./models/cnn_stage_2_phishing_v1.1.keras
3065/3065 [==============================] - 3s 891us/step
📦 Loading model from ./models/XgBoost_stage_2_phishing_v1.1.xgb
📦 Loading model from ./models/Lgbm_stage_2_phishing_v1.1.pkl
📦 Loading model from ./models/feedforwar

,sample_idx,expected,final_pred,decided_at_stage,stage1_votes,stage2_votes,stage3_votes,correct
0,0,0,0,3,0.0,0.0,0.0,1
1,1,0,0,3,0.0,0.0,0.0,1
2,2,0,0,3,0.0,0.0,0.0,1
3,3,1,1,1,3.0,NaN,NaN,1
4,4,0,0,3,0.0,0.0,0.0,1
5,5,0,0,3,0.0,0.0,0.0,1
6,6,0,0,3,0.0,0.0,0.0,1
7,7,0,0,3,0.0,0.0,0.0,1
8,8,0,0,3,0.0,0.0,0.0,1
9,9,0,0,3,0.0,0.0,0.0,1


In [ ]:
display(results_df.head(20))

### Save detailed cascade results


In [ ]:

out_prefix = f"sequential_cascade_{MALICIOUS_LABEL}_{'verification' if VERIFICATION else 'validation'}"
results_path = f"./results/{out_prefix}_details.csv"
summary_path = f"./results/{out_prefix}_summary.csv"

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)

print(f"Saved detailed results to: {results_path}")
print(f"Saved summary to: {summary_path}")

